**Scalability Analysis**

18) Strong Scaling Proxy

In [ ]:
# Function to measure how long a model takes to transform a dataset
# It applies the model and forces execution using count()
def timed_transform_count(model, df):
    start = time.time()
    _ = model.transform(df).count()  # Trigger Spark action
    return time.time() - start


# -------------------------
# Strong Scaling Proxy Test
# -------------------------
# Keep dataset size fixed and vary shuffle partitions
# This simulates how task parallelism affects runtime

strong_rows = []

# Use a small fixed sample to keep runtime reasonable
fixed_df = curated_df.sample(False, 0.05, seed=42).cache()
fixed_df.count()  # Materialise cache

# Test different shuffle partition settings
for sp in [10, 50, 200]:
    spark.conf.set("spark.sql.shuffle.partitions", str(sp))

    # Measure execution time of best model
    dur = timed_transform_count(best_model, fixed_df)

    strong_rows.append({
        "shuffle_partitions": sp,
        "rows": fixed_df.count(),
        "runtime_sec": dur
    })

# Clear cache after experiment
fixed_df.unpersist()

# Convert results to pandas for analysis
strong_pdf = pd.DataFrame(strong_rows)

# Simple cost proxy (runtime × assumed core factor)
strong_pdf["cost_proxy"] = strong_pdf["runtime_sec"] * 2

# Save results for reporting
strong_pdf.to_csv("strong_scaling_proxy.csv", index=False)

print("Saved: strong_scaling_proxy.csv")
strong_pdf

19) Weak Scaling Proxy

In [ ]:
# -------------------------
# Weak Scaling Proxy Test
# -------------------------
# Increase dataset size while keeping configuration fixed
# This observes how runtime grows as data volume increases

weak_rows = []

for frac in [0.01, 0.03, 0.05, 0.10]:

    # Take a fraction of the dataset
    df_frac = curated_df.sample(False, frac, seed=42).cache()

    # Count rows to record actual dataset size
    rows = df_frac.count()

    # Measure execution time of best model on this subset
    dur = timed_transform_count(best_model, df_frac)

    weak_rows.append({
        "fraction": frac,
        "rows": rows,
        "runtime_sec": dur
    })

    # Clear cache after each iteration
    df_frac.unpersist()

# Convert results to pandas
weak_pdf = pd.DataFrame(weak_rows)

# Simple cost proxy for discussion (runtime × assumed core factor)
weak_pdf["cost_proxy"] = weak_pdf["runtime_sec"] * 2

# Save results for scalability analysis in report
weak_pdf.to_csv("weak_scaling_proxy.csv", index=False)

print("Saved: weak_scaling_proxy.csv")
weak_pdf